# Quantum-circuit metrics

Reproduces SI Figure S2 (charge 3+, 34-50 qubits): depth, two-qubit depth, gate count and two-qubit gate count of the transpiled LUCJ circuits, comparing the default Qiskit transpilation (`../quantum_circuits/<medium>/charge<C>_spin<S>/transpiler_pass_initial/`) with Q-CTRL Fire Opal (`../quantum_circuits/<medium>/charge<C>_spin<S>/fire_opal_circuits/`).

With `RUN_FROM_SCRATCH = True` the metrics are recomputed from the QPY files; missing files are skipped. With `RUN_FROM_SCRATCH = False` the tabulated metrics in the cell below are used.

### Imports

In [ ]:
import qiskit

qiskit.__version__

In [ ]:
import numpy as np
import pickle as pkl
from pathlib import Path
from collections import OrderedDict
import matplotlib.pyplot as plt
from qiskit import qpy
from itertools import product

### Setup:

In [ ]:
shifts = ['-0.15','0.0','0.15','0.3','0.45','0.6','0.75','0.9','1.05','1.2']
CIRC_MAIN_DIR = Path("../quantum_circuits")

### Run analysis:

In [ ]:
def clean_count_ops(ops_counter: OrderedDict):
    if 'delay' in ops_counter:
        ops_counter.pop('delay')
    if 'barrier' in ops_counter:
        ops_counter.pop('barrier')
    if 'measure' in ops_counter:
        ops_counter.pop('measure')

    return ops_counter

def analyze_circuits(qcs):
    results = {
        "depth": [],
        "2q-depth": [],
        "num_gates": [],
        "num_2q-gates": [],
    }
    for qc in qcs:
        results["depth"].append(qc.depth())
        results["2q-depth"].append(qc.depth(filter_function=lambda x: x.operation.num_qubits==2))
        
        clean_counter = clean_count_ops(qc.count_ops())
        results["num_gates"].append(sum(clean_counter.values()))
       
        clean_counter = clean_count_ops(qc.count_ops())
        num_2q_gates = 0
        if 'cz' in clean_counter:
            num_2q_gates += clean_counter.get('cz', 0)
        if 'rzz' in clean_counter:
            num_2q_gates += clean_counter.get('rzz', 0)
        if 'cx' in clean_counter:
            num_2q_gates += clean_counter.get('cx', 0)
        results["num_2q-gates"].append(num_2q_gates)
    
    return results

def load_qc_qpy(qc_fname):
    with open(qc_fname, "rb") as f:
        qc = qpy.load(f)
    return qc

def load_qcs_for_shifts_qctrl(shifts, medium, charge, spin, num_qubits, backend_name):
    qcs_list = []
    for shift in shifts:
        qc_fname = CIRC_MAIN_DIR / f"{medium}/charge{charge}_spin{spin}/fire_opal_circuits/qubits_{num_qubits}_shift_{shift}_{backend_name}_no_frac_gates.qpy"

        if qc_fname.exists():
            qc = load_qc_qpy(qc_fname)
            if len(qc) == 1:
                qcs_list.append(qc[0])
            else:
                raise ValueError(f"Expected exactly one circuit in {qc_fname}, but found {len(qc)}")
    
    return qcs_list

def load_qcs_for_shifts_ibm(shifts, medium, charge, spin, num_qubits, backend_name):
    qcs_list = []
    for shift in shifts:
        qc_fname = CIRC_MAIN_DIR / f"{medium}/charge{charge}_spin{spin}/transpiler_pass_initial/{num_qubits}_qubits/qubits_{num_qubits}_shift_{shift}_{backend_name}.qpy"

        if qc_fname.exists():
            qc = load_qc_qpy(qc_fname)
            if len(qc) == 1:
                qcs_list.append(qc[0])
            else:
                raise ValueError(f"Expected exactly one circuit in {qc_fname}, but found {len(qc)}")
    
    return qcs_list

In [ ]:
charge = 3
backend_name = 'ibm_pittsburgh'
media = ['gas', 'solvent']
spins = [0, 4]
num_qubits_list = [34, 38, 46, 50]

RUN_FROM_SCRATCH = False

if RUN_FROM_SCRATCH:
    qctrl_results = {medium: {spin: {num_qubits: None for num_qubits in num_qubits_list} for spin in spins} for medium in media}

    for medium, spin, num_qubits in product(
        media,
        spins,
        num_qubits_list,
    ):
        print(f"Processing medium={medium}, spin={spin}, num_qubits={num_qubits}, backend_name={backend_name}")
        try:
            qcs_list = load_qcs_for_shifts_qctrl(shifts, medium, charge, spin, num_qubits, backend_name)
            qctrl_results[medium][spin][num_qubits] = analyze_circuits(qcs_list) 
        except Exception as e:
            print(f"Skipping processing medium={medium}, spin={spin}, num_qubits={num_qubits}, backend_name={backend_name}: {e}")
            
    default_results = {medium: {spin: {num_qubits: None for num_qubits in num_qubits_list} for spin in spins} for medium in media}

    for medium, spin, num_qubits in product(
        media,
        spins,
        num_qubits_list,
    ):
        print(f"Processing medium={medium}, spin={spin}, num_qubits={num_qubits}, backend_name={backend_name}")
        try:
            qcs_list = load_qcs_for_shifts_ibm(shifts, medium, charge, spin, num_qubits, backend_name)
            default_results[medium][spin][num_qubits] = analyze_circuits(qcs_list) 
        except Exception as e:
            print(f"Skipping processing medium={medium}, spin={spin}, num_qubits={num_qubits}, backend_name={backend_name}: {e}")
else:
    
    qctrl_results = {'gas': {0: {34: {'depth': [387, 314, 275, 308, 299, 308, 284, 228, 241, 260],
    '2q-depth': [116, 98, 81, 94, 93, 89, 83, 71, 74, 79],
    'num_gates': [4611, 4428, 4225, 4305, 4388, 4131, 4177, 3932, 4083, 4146],
    'num_2q-gates': [891, 823, 814, 828, 846, 806, 799, 764, 784, 774]},
   38: {'depth': [329, 413, 302, 359, 312, 377, 357, 450, 317, 273],
    '2q-depth': [98, 128, 96, 112, 91, 117, 112, 135, 95, 85],
    'num_gates': [5658, 5650, 5199, 5171, 5268, 5538, 5658, 5370, 5125, 4941],
    'num_2q-gates': [1097, 1067, 1003, 1001, 1001, 1026, 1022, 1006, 976, 966]},
   46: {'depth': [394, 387, 441, 442, 448, 410, 427, 332, 480, 513],
    '2q-depth': [120, 119, 133, 132, 131, 126, 125, 104, 155, 167],
    'num_gates': [7671, 7894, 7794, 7865, 7559, 7556, 8059, 7423, 7986, 8346],
    'num_2q-gates': [1566, 1569, 1531, 1528, 1498, 1490, 1549, 1523, 1574, 1569]},
   50: {'depth': [620, 468, 423, 460, 444, 408, 423, 459, 424, 364],
    '2q-depth': [189, 143, 130, 138, 130, 122, 122, 137, 128, 107],
    'num_gates': [10592, 9923, 9586, 9031, 9670, 9174, 9232, 8992, 9227, 8669],
    'num_2q-gates': [1949, 1924, 1888, 1826, 1840, 1834, 1817, 1774, 1780, 1719]}},
  4: {34: {'depth': [296, 259, 354, 325, 265, 318, 245, 264, 400, 394],
    '2q-depth': [88, 81, 105, 94, 84, 93, 75, 81, 118, 120],
    'num_gates': [4179, 4434, 4633, 4457, 4170, 4448, 4139, 4331, 4482, 4420],
    'num_2q-gates': [852, 872, 876, 864, 837, 858, 825, 850, 854, 862]},
   38: {'depth': [361, 276, 346, 358, 304, 404, 288, 317, 409, 418],
    '2q-depth': [117, 82, 104, 105, 93, 129, 89, 94, 135, 126],
    'num_gates': [5268, 4864, 5458, 5461, 5137, 5530, 5091, 5343, 5266, 5217],
    'num_2q-gates': [1059, 1023, 1042, 1037, 1049, 1062, 1027, 1021, 1042, 1006]},
   46: {'depth': [460, 452, 441, 482, 353, 332, 362, 301, 306, 350],
    '2q-depth': [141, 135, 134, 157, 105, 104, 109, 90, 94, 107],
    'num_gates': [8238, 8157, 8520, 7772, 7406, 7296, 7556, 7044, 7102, 7282],
    'num_2q-gates': [1641, 1610, 1651, 1579, 1559, 1519, 1538, 1515, 1524, 1534]},
   50: None}},
 'solvent': {0: {34: {'depth': [248, 249, 269, 248, 246, 241, 370, 252, 297, 232],
    '2q-depth': [75, 72, 80, 72, 71, 70, 109, 74, 93, 69],
    'num_gates': [4184, 4067, 4163, 4138, 4014, 3972, 4381, 3972, 4085, 3930],
    'num_2q-gates': [834, 798, 798, 794, 795, 784, 800, 762, 772, 763]},
   38: {'depth': [442, 319, 357, 292, 332, 419, 330, 431, 332, 285],
    '2q-depth': [139, 94, 103, 89, 98, 126, 97, 126, 94, 82],
    'num_gates': [5945, 5331, 5217, 5179, 5641, 5375, 5210, 5269, 5159, 4880],
    'num_2q-gates': [1086, 1022, 1011, 1002, 1030, 1032, 995, 971, 984, 964]},
   46: {'depth': [419, 406, 455, 426, 545, 437, 440, 448, 367, 595],
    '2q-depth': [125, 122, 134, 128, 165, 129, 132, 131, 116, 183],
    'num_gates': [8331, 7928, 7949, 7566, 8213, 7838, 8083, 8077, 7720, 8236],
    'num_2q-gates': [1639, 1563, 1541, 1502, 1573, 1527, 1594, 1548, 1530, 1540]},
   50: {'depth': [601, 476, 381, 453, 524, 582, 377, 445, 371, 410],
    '2q-depth': [183, 148, 117, 134, 156, 174, 117, 133, 113, 128],
    'num_gates': [10025, 9688, 9027, 9477, 8930, 9627, 8661, 9186, 8920, 9219],
    'num_2q-gates': [1909, 1894, 1864, 1907, 1820, 1847, 1802, 1806, 1770, 1795]}},
  4: {34: {'depth': [244, 298, 234, 263, 243, 378, 229, 312, 265, 262],
    '2q-depth': [74, 94, 72, 82, 74, 114, 71, 93, 79, 80],
    'num_gates': [4132, 4312, 4003, 4311, 3932, 4616, 3989, 4308, 4441, 4261],
    'num_2q-gates': [844, 857, 841, 846, 817, 866, 821, 838, 844, 831]},
   38: {'depth': [370, 467, 363, 303, 299, 368, 278, 431, 272, 397],
    '2q-depth': [119, 140, 117, 91, 90, 114, 86, 136, 81, 126],
    'num_gates': [5191, 5877, 5185, 5235, 5123, 5200, 5005, 5527, 4991, 5393],
    'num_2q-gates': [1045, 1096, 1057, 1043, 1049, 1041, 1015, 1053, 1016, 1044]},
   46: {'depth': [481, 353, 389, 422, 488, 489, 323, 415, 470, 434],
    '2q-depth': [139, 107, 121, 126, 152, 159, 101, 123, 154, 126],
    'num_gates': [8147, 7799, 7935, 7936, 7995, 7559, 7224, 7471, 7497, 7547],
    'num_2q-gates': [1624, 1623, 1643, 1606, 1559, 1520, 1533, 1529, 1532, 1528]},
   50: {'depth': [512, 537, 334, 321, 318],
    '2q-depth': [160, 164, 104, 101, 103],
    'num_gates': [9422, 9483, 8239, 8195, 8143],
    'num_2q-gates': [1906, 1930, 1838, 1806, 1822]}}}}
    
    default_results = {
        'gas': {
            0: {
                34: {
                    'depth': [459, 389, 366, 386, 447, 367, 359, 348, 345, 350],
                    '2q-depth': [132, 105, 102, 105, 122, 97, 99, 93, 93, 94],
                    'num_gates': [6786, 6683, 6616, 6534, 6607, 6582, 6568, 6500, 6449, 6462],
                    'num_2q-gates': [1038, 1010, 1012, 1014, 1030, 996, 1010, 996, 988, 994]
                },
                38: {
                    'depth': [451, 577, 412, 569, 511, 575, 439, 596, 387, 366],
                    '2q-depth': [123, 146, 112, 146, 148, 144, 122, 151, 112, 100],
                    'num_gates': [8431, 8315, 8167, 8127, 8192, 8187, 8216, 7996, 8017, 8034],
                    'num_2q-gates': [1296, 1283, 1238, 1227, 1266, 1245, 1254, 1206, 1204, 1208]
                },
                46: {
                    'depth': [562, 505, 661, 669, 658, 584, 625, 451, 701, 678],
                    '2q-depth': [161, 141, 189, 192, 186, 160, 176, 125, 180, 177],
                    'num_gates': [12475, 12051, 11975, 12027, 11943, 11934, 11872, 11719, 11664, 11795],
                    'num_2q-gates': [1894, 1818, 1791, 1807, 1821, 1794, 1809, 1757, 1754, 1788]
                },
                50: {
                    'depth': [760, 578, 523, 533, 537, 513, 545, 573, 519, 501],
                    '2q-depth': [217, 176, 148, 147, 157, 137, 165, 158, 143, 131],
                    'num_gates': [14824, 14772, 14401, 14229, 14333, 14295, 14319, 13989, 13878, 13767],
                    'num_2q-gates': [2272, 2257, 2158, 2134, 2172, 2153, 2198, 2138, 2136, 2126]
                }
            },
            4: {
                34: {
                    'depth': [387, 381, 466, 404, 370, 393, 368, 358, 467, 479],
                    '2q-depth': [101, 101, 126, 118, 103, 119, 103, 97, 125, 132],
                    'num_gates': [6669, 6726, 6709, 6724, 6680, 6711, 6660, 6657, 6694, 6662],
                    'num_2q-gates': [995, 1011, 1010, 1021, 996, 1011, 990, 986, 1010, 1000]
                },
                38: {
                    'depth': [582, 392, 508, 514, 383, 571, 374, 472, 577, 515],
                    '2q-depth': [148, 106, 143, 138, 110, 149, 103, 128, 150, 140],
                    'num_gates': [8340, 8270, 8349, 8407, 8210, 8223, 8150, 8204, 8135, 8161],
                    'num_2q-gates': [1244, 1232, 1260, 1261, 1202, 1206, 1186, 1205, 1191, 1199]
                },
                46: {
                    'depth': [687, 653, 534, 695, 452, 437, 440, 422, 418, 454],
                    '2q-depth': [194, 184, 167, 185, 120, 121, 125, 111, 111, 122],
                    'num_gates': [12292, 12226, 12319, 11974, 11948, 11826, 11858, 11771, 11721, 11739],
                    'num_2q-gates': [1864, 1831, 1862, 1736, 1732, 1690, 1705, 1685, 1674, 1676]
                },
                50: {
                    'depth': [527, 552, 620, 679, 492, 466, 485, 465, 476, 474],
                    '2q-depth': [142, 151, 170, 184, 130, 121, 125, 121, 125, 127],
                    'num_gates': [14362, 14299, 14504, 14090, 13975, 13945, 13912, 13945, 13876, 13950],
                    'num_2q-gates': [2116, 2100, 2166, 2031, 2000, 1980, 1976, 1982, 1968, 1976]
                }
            }
        },
        'solvent': {
            0: {
                34: {
                    'depth': [362, 351, 367, 359, 359, 343, 468, 366, 376, 353],
                    '2q-depth': [101, 92, 99, 96, 98, 93, 130, 100, 103, 94],
                    'num_gates': [6782, 6611, 6576, 6599, 6633, 6578, 6637, 6606, 6493, 6552],
                    'num_2q-gates': [1026, 1000, 1006, 1000, 1009, 998, 1024, 1008, 988, 995]
                },
                38: {
                    'depth': [641, 402, 507, 478, 532, 556, 491, 509, 499, 366],
                    '2q-depth': [173, 112, 140, 131, 157, 150, 136, 139, 139, 99],
                    'num_gates': [8353, 8338, 8294, 8241, 8280, 8232, 8122, 8073, 8071, 7952],
                    'num_2q-gates': [1277, 1275, 1268, 1260, 1266, 1251, 1220, 1213, 1220, 1196]
                },
                46: {
                    'depth': [650, 654, 659, 631, 629, 619, 539, 616, 469, 747],
                    '2q-depth': [188, 186, 186, 178, 180, 175, 151, 173, 127, 217],
                    'num_gates': [12153, 11983, 12030, 12014, 12002, 11919, 11816, 11769, 11826, 11846],
                    'num_2q-gates': [1874, 1806, 1799, 1802, 1823, 1806, 1797, 1792, 1790, 1805]
                },
                50: {
                    'depth': [726, 605, 508, 541, 700, 681, 538, 527, 475, 524],
                    '2q-depth': [206, 173, 137, 163, 195, 189, 146, 144, 129, 144],
                    'num_gates': [14699, 14594, 14328, 14459, 14363, 14326, 14267, 14137, 13891, 13804],
                    'num_2q-gates': [2234, 2202, 2132, 2214, 2158, 2174, 2155, 2172, 2132, 2136]
                }
            },
            4: {
                34: {
                    'depth': [407, 393, 351, 367, 351, 485, 352, 356, 359, 347],
                    '2q-depth': [111, 115, 93, 103, 94, 130, 93, 94, 98, 98],
                    'num_gates': [6719, 6763, 6643, 6671, 6585, 6607, 6648, 6480, 6499, 6411],
                    'num_2q-gates': [1005, 1024, 984, 990, 984, 1002, 985, 972, 981, 980]
                },
                38: {
                    'depth': [569, 577, 590, 495, 386, 577, 377, 591, 371, 548],
                    '2q-depth': [141, 161, 153, 138, 105, 150, 103, 147, 99, 137],
                    'num_gates': [8282, 8478, 8358, 8417, 8277, 8116, 8104, 8107, 8030, 8006],
                    'num_2q-gates': [1232, 1277, 1240, 1268, 1234, 1196, 1181, 1197, 1176, 1185]
                },
                46: {
                    'depth': [678, 510, 528, 648, 725, 660, 439, 557, 661, 561],
                    '2q-depth': [198, 139, 150, 186, 181, 164, 122, 155, 164, 155],
                    'num_gates': [12306, 12311, 12231, 12219, 11956, 11763, 11827, 11766, 11781, 11814],
                    'num_2q-gates': [1851, 1836, 1828, 1826, 1749, 1677, 1702, 1706, 1687, 1698]
                },
                50: {
                    'depth': [694, 528, 688, 702, 488, 464, 476, 466, 650, 463],
                    '2q-depth': [190, 148, 190, 192, 128, 122, 126, 121, 177, 121],
                    'num_gates': [14415, 14390, 14521, 14434, 14001, 13953, 13960, 13943, 14013, 13962],
                    'num_2q-gates': [2133, 2116, 2165, 2144, 2000, 1984, 1984, 1986, 2005, 1984]
                }
            }
        }
    }

In [ ]:
import shutil
import scienceplots
# 'science' style renders text with LaTeX; fall back to mathtext if no LaTeX installation is found
STYLE = ['science'] if shutil.which('latex') else ['science', 'no-latex']

distance = [1.44, 1.97, 2.51, 3.04, 3.57, 4.10, 4.63, 5.16, 5.69, 6.23]
shift_array = ['-0.15','0.0','0.15','0.3','0.45','0.6','0.75','0.9','1.05','1.2']


colors = ['#348ABD', '#A60628', '#7A68A6', '#467821', '#D55E00', '#CC79A7', '#56B4E9', '#009E73', '#F0E442', '#0072B2']
sign = ['o', 'v']

# with plt.style.context('bmh'):
alpha = 0.22
with plt.style.context(STYLE):
    for medium in media:
    
        fig, axs = plt.subplots(1, 4, figsize=(13, 3.), dpi=200)
        for i, metric in enumerate(['depth', '2q-depth', 'num_gates', 'num_2q-gates']):
            # Collect all values for this metric across all configurations
            for j, spin in enumerate(spins):
                all_values = []
                x_values = []
                for nq in num_qubits_list:
                    if qctrl_results[medium][spin][nq] is not None:
                        
                        all_values.append(qctrl_results[medium][spin][nq][metric])
                        x_values.append(nq)

                if all_values:
                    # Convert to numpy array and compute mean and min-max
                    mean_vals = np.array([np.mean(vals) for vals in all_values])
                    min_vals = np.array([np.min(vals) for vals in all_values])
                    max_vals = np.array([np.max(vals) for vals in all_values])
                    
                    
                    # Plot mean line
                    axs[i].plot(x_values, mean_vals, '-' + sign[j], color=colors[j], linewidth=2, label='Spin ' + str(spin//2))
                    
                    # Plot min-max spread
                    axs[i].fill_between(x_values, min_vals, max_vals, alpha=alpha, color=colors[j], label=None)
                    
                # Do the same for default results
                default_values = []
                x_values_default = []
                for nq in num_qubits_list:
                    if default_results[medium][spin][nq] is not None:
                        default_values.append(default_results[medium][spin][nq][metric])
                        x_values_default.append(nq)
                
                if default_values:
                    default_values = np.array(default_values)
                    default_mean = np.mean(default_values, axis=1)
                    default_min = np.min(default_values, axis=1)
                    default_max = np.max(default_values, axis=1)
                    
                    axs[i].plot(x_values_default, default_mean, '--' + sign[j], color=colors[j], linewidth=2, label=None)
                    axs[i].fill_between(x_values_default, default_min, default_max, alpha=alpha, color=colors[j], label=None)
                
            axs[i].grid(True)
            axs[i].set_xlabel("Num. qubits")
            axs[i].set_title(metric.replace("_", ". ").capitalize())
            axs[i].set_ylim(bottom=0)
                
        axs[-1].plot([], [], 'k-', label='Q-CTRL')
        axs[-1].plot([], [], 'k--', label='default')
        
        axs[-1].plot([], [], 'k-', label='mean')
        axs[-1].fill_between([], [], color='black', label='min-max', alpha=alpha)
        axs[-1].set_ylim(bottom=0)

        
        axs[-1].legend(frameon=True)
        plt.suptitle(f"Circuit metrics charge +3, IBM {backend_name.split('_')[-1].capitalize()} - {medium}", y=1.04, fontsize=14)
        # plt.tight_layout()

        plt.savefig(f"./figures/qc_metrics_charge{charge}_{medium}.png", dpi=300, bbox_inches='tight')
        plt.show()

In [ ]:
import shutil
import scienceplots
# 'science' style renders text with LaTeX; fall back to mathtext if no LaTeX installation is found
STYLE = ['science'] if shutil.which('latex') else ['science', 'no-latex']

distance = [1.44, 1.97, 2.51, 3.04, 3.57, 4.10, 4.63, 5.16, 5.69, 6.23]
shift_array = ['-0.15','0.0','0.15','0.3','0.45','0.6','0.75','0.9','1.05','1.2']


colors = ['#348ABD', '#A60628', '#7A68A6', '#467821', '#D55E00', '#CC79A7', '#56B4E9', '#009E73', '#F0E442', '#0072B2']
for nq_i, nq in enumerate(num_qubits_list):
    # with plt.style.context('bmh'):
    with plt.style.context(STYLE):
        fig, axs = plt.subplots(1, 4, figsize=(14, 3))
        for m_s_i, (medium, spin) in enumerate(product(media, spins)):
            if qctrl_results[medium][spin][nq] is None or len(qctrl_results[medium][spin][nq]['depth']) != len(distance):
                print(f"Skipping plotting for medium={medium}, spin={spin}, num_qubits={nq} due to missing data")
                continue
            for i, (k, v) in enumerate(qctrl_results[medium][spin][nq].items()):
                axs[i].plot(distance, v, '-', label=f"{medium}, spin {spin}", color=colors[m_s_i])
                axs[i].plot(distance, default_results[medium][spin][nq][k], '--', label=None, color=colors[m_s_i])

                # if m_s_i == 0:
                axs[i].grid(True)
                axs[i].set_xlabel("Distance [A]")
                axs[i].set_title(k.replace("_", ". ").capitalize())
                if axs[i].get_ylim()[-1] < max( default_results[medium][spin][nq][k])*1.1:
                    axs[i].set_ylim(bottom=0, top=max( default_results[medium][spin][nq][k])*1.1)
                else:
                    axs[i].set_ylim(bottom=0)
        
        axs[0].plot([], [], 'k-', color='k', label='Q-CTRL')
        axs[0].plot([], [], 'k--', color='k', label='default')
        
        axs[0].legend(frameon=True)

        plt.suptitle(f"Circuit metrics charge +3, IBM {backend_name.split('_')[-1].capitalize()}", y=1.01)
        plt.show()